# Parquet → Neo4j CSV — interactive validator

Convert each RP parquet dim/fact to a `neo4j-admin import` CSV **one table at a time**
and validate that the generated CSV matches the source parquet (row count, unique
keys, sample rows). The final two sections generate the bulk import script and
compare the live Neo4j graph against the source parquets.

**How to use:** run cells top-to-bottom for setup, then jump into any table
section. Each table has an *Export* cell and a *Validate* cell that you can
re-run independently.


## 1. Setup


In [ ]:
import sys, os, time, logging
from pathlib import Path
import pandas as pd

# Quiet pandas DtypeWarning in mixed-type columns
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s — %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('export')


In [ ]:
# Paths — adjust to your environment
DATA_DIR = Path('/home/siva/work/codebase/RP/synthea-neo4j/data/rp_synthetic_output')
OUT_DIR  = Path('/home/siva/work/codebase/RP/synthea-neo4j/notebooks/neo4j_import')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR :', DATA_DIR, '(exists)' if DATA_DIR.exists() else '(MISSING)')
print('OUT_DIR  :', OUT_DIR)


## 2. Helpers (composite IDs + CSV writer)


In [ ]:
def clean(df):
    """NaN/NaT/None -> empty string for CSV output."""
    return df.where(pd.notnull(df), "")

def fmt_dt(series):
    """Datetime columns -> ISO strings neo4j-admin understands."""
    return pd.to_datetime(series, errors="coerce").dt.strftime("%Y-%m-%dT%H:%M:%S").fillna("")

# Composite ID builders
def pid(src, x): return src.astype(str) + ":" + x.astype(str)  # Patient
def lid(src, x): return src.astype(str) + ":" + x.astype(str)  # Location
def vid(src, x): return src.astype(str) + ":" + x.astype(str)  # Visit
def cid(src, x): return src.astype(str) + ":" + x.astype(str)  # Charge
def tid(src, x): return src.astype(str) + ":" + x.astype(str)  # Transaction
def iid(src, x): return src.astype(str) + ":" + x.astype(str)  # InsurancePlan

def write_csv(df, path, label):
    df.to_csv(path, index=False)
    print(f"  ✓ {path.name:<45} {len(df):>10,} rows   [{label}]")
    return df


### 2.1 Validation helper

`validate(label, src_count, csv_path, id_col=None)` compares the source parquet
row count with the generated CSV and (optionally) checks the ID column is unique.


In [ ]:
def validate(label, source_df, csv_path, id_col=None, allow_drop=False):
    """Compare source parquet vs generated CSV.
       - source_df: the parquet dataframe (or row count int)
       - csv_path:  path to the generated CSV
       - id_col:    column expected to be unique (the ":ID(Label)" column)
       - allow_drop: if True, source > csv is OK (e.g., after deduplication)
    """
    src_n = source_df if isinstance(source_df, int) else len(source_df)
    csv = pd.read_csv(csv_path, low_memory=False)
    csv_n = len(csv)
    delta = csv_n - src_n
    pct   = (delta / src_n * 100) if src_n else 0.0

    ok = "✓" if (delta == 0 or (allow_drop and delta <= 0)) else "✗"
    print(f"{ok} {label}")
    print(f"   source rows: {src_n:>10,}")
    print(f"   csv rows   : {csv_n:>10,}  (Δ {delta:+,} / {pct:+.2f}%)")

    if id_col and id_col in csv.columns:
        nun = csv[id_col].nunique(dropna=False)
        dup = csv_n - nun
        bad = "✗" if dup > 0 else "✓"
        print(f"   {bad} {id_col}: {nun:,} unique, {dup:,} duplicates")

    print("\n  Sample of CSV:")
    return csv.head(3)


## 3. Nodes (16 tables)

Each section: **load parquet → build CSV → write → validate**.

### 3.1 Patient


In [ ]:
# Load patient dims + nav map for financial enrichment
pat = pd.read_parquet(DATA_DIR / '02_dims/patient.parquet')
nav_cols = ["Source_Database_Code","PatientID","total_charged","total_paid","outstanding_balance","total_adjusted",
            "adj_contractual","adj_bad_debt","adj_collection_agency","adj_refund_reversal","adj_other",
            "payor_cohort","call_tier","is_catastrophe","is_friction","is_clean","is_self_pay","is_bai",
            "is_fully_covered","is_sapa","is_nraa","is_tennessee","is_atlanta_404",
            "multi_practice_flag","practice_count","visit_count","charge_count","statement_count",
            "total_calls_window","has_any_calls","has_insurance","_active_window",
            "PlanName","Carrier_Name","PlanType"]
nav = pd.read_parquet(DATA_DIR / '00_navigation/patient_navigation_map.parquet', columns=nav_cols).drop_duplicates(subset=["Source_Database_Code","PatientID"], keep="last")
df = pat.merge(nav, on=["Source_Database_Code","PatientID"], how="left")

out = pd.DataFrame()
out["patientId:ID(Patient)"]       = pid(df["Source_Database_Code"], df["PatientID"])
out["source_db"]                   = df["Source_Database_Code"]
out["patient_id:long"]             = df["PatientID"]
out["first_name"]                  = df["PatientFirstName"].fillna("")
out["last_name"]                   = df["PatientLastName"].fillna("")
out["dob:datetime"]                = fmt_dt(df["PatientDOB"])
out["gender"]                      = df["PatientGender"].fillna("")
out["zip"]                         = df["PatientZip"].fillna("")
out["phone_norm"]                  = df["PatientPhone_norm"].fillna("")
out["email"]                       = df["patEmail"].fillna("")
out["total_charged:float"]         = df["total_charged"].fillna("")
out["total_paid:float"]            = df["total_paid"].fillna("")
out["outstanding_balance:float"]   = df["outstanding_balance"].fillna("")
out["adj_bad_debt:float"]          = df["adj_bad_debt"].fillna("")
out["payor_cohort"]                = df["payor_cohort"].fillna("")
out["call_tier"]                   = df["call_tier"].fillna("")
out["is_catastrophe:boolean"]      = df["is_catastrophe"].fillna(False)
out["is_self_pay:boolean"]         = df["is_self_pay"].fillna(False)
out["practice_count:int"]          = df["practice_count"].fillna(0)
out["visit_count:float"]           = df["visit_count"].fillna(0)
out["charge_count:int"]            = df["charge_count"].fillna(0)
out["carrier_name"]                = df["Carrier_Name"].fillna("")
out["plan_name"]                   = df["PlanName"].fillna("")
out[":LABEL"]                      = "Patient"
write_csv(out, OUT_DIR / 'nodes_patient.csv', 'Patient')


In [ ]:
validate('Patient', len(pat), OUT_DIR / 'nodes_patient.csv', id_col='patientId:ID(Patient)')

### 3.2 Practice


In [ ]:
df = pd.read_parquet(DATA_DIR / '02_dims/patient.parquet', columns=["Source_Database_Code"])
codes = df["Source_Database_Code"].dropna().unique()
out = pd.DataFrame({
    "practiceId:ID(Practice)": codes,
    "code": codes,
    ":LABEL": "Practice",
})
write_csv(out, OUT_DIR / 'nodes_practice.csv', 'Practice')


In [ ]:
validate('Practice', len(codes), OUT_DIR / 'nodes_practice.csv', id_col='practiceId:ID(Practice)')

### 3.3 Location


In [ ]:
df = pd.read_parquet(DATA_DIR / '02_dims/location.parquet')
out = pd.DataFrame()
out["locationId:ID(Location)"]    = lid(df["Source_Database_Code"], df["LocationID"])
out["source_db"]                  = df["Source_Database_Code"]
out["name"]                       = df["LocationName"].fillna("")
out["address"]                    = df["LocationAddress"].fillna("")
out["city"]                       = df["LocationCity"].fillna("")
out["state"]                      = df["LocationState"].fillna("")
out["zip"]                        = df["LocationZip"].fillna("")
out["phone_norm"]                 = df["LocationPhone_norm"].fillna("")
out["location_type"]              = df["LocationType"].fillna("")
out["birdeye_review_count:float"] = df["birdeye_review_count"].fillna("")
out["birdeye_avg_rating:float"]   = df["birdeye_avg_rating"].fillna("")
out[":LABEL"]                     = "Location"
write_csv(out, OUT_DIR / 'nodes_location.csv', 'Location')


In [ ]:
validate('Location', len(df), OUT_DIR / 'nodes_location.csv', id_col='locationId:ID(Location)')

### 3.4 InsurancePlan


In [ ]:
df = pd.read_parquet(DATA_DIR / '02_dims/insurance.parquet')
out = pd.DataFrame()
out["insuranceId:ID(InsurancePlan)"] = iid(df["Source_Database_Code"], df["PlanNumber"])
out["source_db"]   = df["Source_Database_Code"]
out["plan_number"] = df["PlanNumber"].fillna("")
out["plan_name"]   = df["PlanName"].fillna("")
out["plan_type"]   = df["PlanType"].fillna("")
out["carrier_name"]= df["Carrier_Name"].fillna("")
out[":LABEL"]      = "InsurancePlan"
write_csv(out, OUT_DIR / 'nodes_insurance.csv', 'InsurancePlan')


In [ ]:
validate('InsurancePlan', len(df), OUT_DIR / 'nodes_insurance.csv', id_col='insuranceId:ID(InsurancePlan)')

### 3.5 Campaign


In [ ]:
df = pd.read_parquet(DATA_DIR / '03_supplementary/campaign_map.parquet')
out = pd.DataFrame()
out["campaignId:ID(Campaign)"] = df["Campaign Name"]
out["name"]                    = df["Campaign Name"]
out["source_db"]               = df["Source Database Code"].fillna("")
out["notes"]                   = df["Notes"].fillna("")
out[":LABEL"]                  = "Campaign"
write_csv(out, OUT_DIR / 'nodes_campaign.csv', 'Campaign')


In [ ]:
validate('Campaign', len(df), OUT_DIR / 'nodes_campaign.csv', id_col='campaignId:ID(Campaign)')

### 3.6 BirdeyeReview


In [ ]:
df = pd.read_parquet(DATA_DIR / '02_dims/birdeye.parquet')
src_n = len(df)
df = df.drop_duplicates(subset=["Location","Date Posted On"], keep="last")
bid = df["Location"].astype(str) + "::" + df["Date Posted On"].astype(str)
out = pd.DataFrame()
out["birdeyeId:ID(BirdeyeReview)"] = bid
out["location"]    = df["Location"]
out["date_posted"] = df["Date Posted On"]
out["source"]      = df["Review Source"]
out["rating:int"]  = df["Review Rating"]
out["comment"]     = df["Review Comment"].fillna("")
out["phi_phone:int"]   = df["phi_phone_count"]
out["phi_email:int"]   = df["phi_email_count"]
out["phi_flagged:boolean"] = df["phi_flagged"]
out[":LABEL"] = "BirdeyeReview"
write_csv(out, OUT_DIR / 'nodes_birdeye.csv', 'BirdeyeReview')
print(f"   (dropped {src_n - len(df):,} duplicates by Location+Date)")


In [ ]:
validate('BirdeyeReview', len(df), OUT_DIR / 'nodes_birdeye.csv', id_col='birdeyeId:ID(BirdeyeReview)', allow_drop=True)

### 3.7 Visit


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/visits.parquet')
src_n = len(df)
df = df.drop_duplicates(subset=["Source_Database_Code","VisitID"], keep="last")
out = pd.DataFrame()
out["visitId:ID(Visit)"]        = vid(df["Source_Database_Code"], df["VisitID"])
out["source_db"]                = df["Source_Database_Code"]
out["visit_id"]                 = df["VisitID"].fillna("")
out["admit_date:datetime"]      = fmt_dt(df["AdmitDate"])
out["discharge_date:datetime"]  = fmt_dt(df["DischargeDate"])
out["location_id"]              = df["LocationID"].fillna("")
out["primary_insurance_plan"]   = df["PrimaryInsurancePlanNum"].fillna("")
out[":LABEL"] = "Visit"
write_csv(out, OUT_DIR / 'nodes_visit.csv', 'Visit')
print(f"   (dropped {src_n - len(df):,} duplicates)")


In [ ]:
validate('Visit', len(df), OUT_DIR / 'nodes_visit.csv', id_col='visitId:ID(Visit)', allow_drop=True)

### 3.8 Charge


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet')
out = pd.DataFrame()
out["chargeId:ID(Charge)"]      = cid(df["Source_Database_Code"], df["ChargeID"])
out["source_db"]                = df["Source_Database_Code"]
out["charge_amount:float"]      = df["ChargeAmount"].fillna("")
out["procedure_code"]           = df["ProcedureCode"].fillna("")
out["procedure_description"]    = df["ProcedureDescription"].fillna("")
out["procedure_modality"]       = df["ProcedureModality"].fillna("")
out["service_date:datetime"]    = fmt_dt(df["ServiceDate"])
out["post_date:datetime"]       = fmt_dt(df["PostDate"])
out["balance:float"]            = df["Balance"].fillna("")
out["place_of_service"]         = df["PlaceOfService"].fillna("")
out["line_status"]              = df["LineStatus"].fillna("")
out["icd10_1"]                  = df["ICD10Diagnosis1"].fillna("")
out["icd10_2"]                  = df["ICD10Diagnosis2"].fillna("")
out["icd10_3"]                  = df["ICD10Diagnosis3"].fillna("")
out["patient_id:long"]          = df["PatientID"].fillna("")
out["visit_id"]                 = df["VisitID"].fillna("")
out[":LABEL"] = "Charge"
write_csv(out, OUT_DIR / 'nodes_charge.csv', 'Charge')


In [ ]:
validate('Charge', len(df), OUT_DIR / 'nodes_charge.csv', id_col='chargeId:ID(Charge)')

### 3.9 Transaction


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/transactions.parquet')
out = pd.DataFrame()
out["transactionId:ID(Transaction)"] = tid(df["Source_Database_Code"], df["PaymentID"])
out["source_db"]                = df["Source_Database_Code"]
out["payment_amount:float"]     = df["PaymentAmount"].fillna("")
out["adjustment_amount:float"]  = df["AdjustmentAmount"].fillna("")
out["adjustment_bucket"]        = df["AdjustmentBucket"].fillna("")
out["post_date:datetime"]       = fmt_dt(df["PostDate"])
out["balance_after_post:float"] = df["BalanceAfterPost"].fillna("")
out["paysource"]                = df["Paysource"].fillna("")
out["payment_method"]           = df["PaymentMethod"].fillna("")
out["charge_id"]                = df["ChargeID"].fillna("")
out[":LABEL"] = "Transaction"
write_csv(out, OUT_DIR / 'nodes_transaction.csv', 'Transaction')


In [ ]:
validate('Transaction', len(df), OUT_DIR / 'nodes_transaction.csv', id_col='transactionId:ID(Transaction)')

### 3.10 Statement


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/statements.parquet')
out = pd.DataFrame()
out["statementId:ID(Statement)"] = df["StatementID"]
out["source_db"]                 = df["Source_Database_Code"]
out["patient_balance:float"]     = df["PatientBalance"].fillna("")
out["total_balance:float"]       = df["TotalBalance"].fillna("")
out["statement_level"]           = df["StatementLevel"].fillna("")
out["created_date:datetime"]     = fmt_dt(df["CreatedDate"])
out["is_released:int"]           = df["IsReleased"].fillna(0)
out["patient_id:long"]           = df["PatientID"].fillna("")
out[":LABEL"] = "Statement"
write_csv(out, OUT_DIR / 'nodes_statement.csv', 'Statement')


In [ ]:
validate('Statement', len(df), OUT_DIR / 'nodes_statement.csv', id_col='statementId:ID(Statement)')

### 3.11 RCCall (RingCentral)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/ringcentral.parquet')
out = pd.DataFrame()
out["rccallId:ID(RCCall)"] = df["Contact_ID"]
out["campaign_name"]       = df["Campaign_Name"].fillna("")
out["agent_name"]          = df["Agent_Name"].fillna("")
out["team_name"]           = df["Team_Name"].fillna("")
out["start_date"]          = df["Start_Date"].fillna("")
out["agent_time:int"]      = df["Agent_Time"].fillna(0)
out["total_time:int"]      = df["Total_Time_Plus_Disposition"].fillna(0)
out["disp_name"]           = df["Disp_Name"].fillna("")
out["ani_norm"]            = df["ANI_DIALNUM_norm"].fillna("")
out["rc_attributable:boolean"] = df["_rc_attributable"].fillna(False)
out[":LABEL"] = "RCCall"
write_csv(out, OUT_DIR / 'nodes_rccall.csv', 'RCCall')


In [ ]:
validate('RCCall', len(df), OUT_DIR / 'nodes_rccall.csv', id_col='rccallId:ID(RCCall)')

### 3.12 IVRInbound


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/rv_inbound.parquet')
out = pd.DataFrame()
out["ivrId:ID(IVRInbound)"]   = df["ResponseID"]
out["account_id"]             = df["AccountID"].fillna("")
out["balance:float"]          = df["Balance"].fillna("")
out["amount_paid:float"]      = df["AmountPaid"].fillna("")
out["call_datetime:datetime"] = fmt_dt(df["CallDateTime"])
out["call_duration:float"]    = df["CallDuration"].fillna("")
out["auth_success:boolean"]   = df["AuthenticationSuccess"].fillna(False)
out["result_desc"]            = df["ResultDesc"].fillna("")
out[":LABEL"] = "IVRInbound"
write_csv(out, OUT_DIR / 'nodes_ivrinbound.csv', 'IVRInbound')


In [ ]:
validate('IVRInbound', len(df), OUT_DIR / 'nodes_ivrinbound.csv', id_col='ivrId:ID(IVRInbound)')

### 3.13 DiallerCall


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/rv_outbound.parquet')
out = pd.DataFrame()
out["diallerId:ID(DiallerCall)"] = df["ACCOUNT"]
out["account_id"]                = df["ACCOUNTID"].fillna("")
out["patient_balance:float"]     = df["PATIENTBALANCE"].fillna("")
out["call_datetime:datetime"]    = fmt_dt(df["CALLDATETIME"])
out["result_desc"]               = df["RESULTSDESC"].fillna("")
out["phone_norm"]                = df["PATIENTPHONE_norm"].fillna("")
out[":LABEL"] = "DiallerCall"
write_csv(out, OUT_DIR / 'nodes_diallercall.csv', 'DiallerCall')


In [ ]:
validate('DiallerCall', len(df), OUT_DIR / 'nodes_diallercall.csv', id_col='diallerId:ID(DiallerCall)')

### 3.14 PhoneBridge


In [ ]:
df = pd.read_parquet(DATA_DIR / '03_supplementary/phone_bridge.parquet')
src_n = len(df)
df = df.drop_duplicates(subset=["Source_Database_Code","PatientID","phone_norm"], keep="last")
pbid = (df["Source_Database_Code"].astype(str) + ":" +
        df["PatientID"].astype(str) + ":" +
        df["phone_norm"].astype(str))
out = pd.DataFrame()
out["phonebridgeId:ID(PhoneBridge)"] = pbid
out["source_db"]   = df["Source_Database_Code"]
out["patient_id:long"]   = df["PatientID"]
out["phone_norm"]  = df["phone_norm"].fillna("")
out["phone_type"]  = df["phone_type"].fillna("")
out["rc_call_count:float"] = df["rc_call_count"].fillna(0)
out["primary_campaign"]    = df["primary_campaign"].fillna("")
out[":LABEL"] = "PhoneBridge"
write_csv(out, OUT_DIR / 'nodes_phonebridge.csv', 'PhoneBridge')
print(f"   (dropped {src_n - len(df):,} duplicates)")


In [ ]:
validate('PhoneBridge', len(df), OUT_DIR / 'nodes_phonebridge.csv', id_col='phonebridgeId:ID(PhoneBridge)', allow_drop=True)

### 3.15 DiagnosisCode (distinct ICD10s from charges)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet',
                     columns=["ICD10Diagnosis1","ICD10Diagnosis2","ICD10Diagnosis3","ICD10Diagnosis4","ICD10Diagnosis5"])
codes = pd.concat([df[c] for c in df.columns]).dropna().unique()
out = pd.DataFrame({
    "diagnosisId:ID(DiagnosisCode)": codes,
    "code": codes,
    ":LABEL": "DiagnosisCode",
})
write_csv(out, OUT_DIR / 'nodes_diagnosiscode.csv', 'DiagnosisCode')
print(f"   (distinct codes across 5 ICD columns)")


In [ ]:
validate('DiagnosisCode', len(codes), OUT_DIR / 'nodes_diagnosiscode.csv', id_col='diagnosisId:ID(DiagnosisCode)')

### 3.16 ProcedureCode (distinct from charges)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet',
                     columns=["ProcedureCode","ProcedureDescription","ProcedureModality"])
df = df.dropna(subset=["ProcedureCode"]).drop_duplicates(subset=["ProcedureCode"])
out = pd.DataFrame()
out["procedureId:ID(ProcedureCode)"] = df["ProcedureCode"]
out["code"]        = df["ProcedureCode"]
out["description"] = df["ProcedureDescription"].fillna("")
out["modality"]    = df["ProcedureModality"].fillna("")
out[":LABEL"]      = "ProcedureCode"
write_csv(out, OUT_DIR / 'nodes_procedurecode.csv', 'ProcedureCode')


In [ ]:
validate('ProcedureCode', len(df), OUT_DIR / 'nodes_procedurecode.csv', id_col='procedureId:ID(ProcedureCode)')

## 4. Relationships (22 tables)


### 4.1 REGISTERED_AT (Patient → Practice)


In [ ]:
df = pd.read_parquet(DATA_DIR / '02_dims/patient.parquet', columns=["Source_Database_Code","PatientID"])
out = pd.DataFrame()
out[":START_ID(Patient)"] = pid(df["Source_Database_Code"], df["PatientID"])
out[":END_ID(Practice)"]  = df["Source_Database_Code"]
out[":TYPE"]              = "REGISTERED_AT"
write_csv(out, OUT_DIR / 'rel_patient_practice.csv', 'REGISTERED_AT')


In [ ]:
validate('REGISTERED_AT', len(df), OUT_DIR / 'rel_patient_practice.csv')

### 4.2 BELONGS_TO_PRACTICE (Location → Practice)


In [ ]:
df = pd.read_parquet(DATA_DIR / '02_dims/location.parquet', columns=["Source_Database_Code","LocationID"])
out = pd.DataFrame()
out[":START_ID(Location)"] = lid(df["Source_Database_Code"], df["LocationID"])
out[":END_ID(Practice)"]   = df["Source_Database_Code"]
out[":TYPE"]               = "BELONGS_TO_PRACTICE"
write_csv(out, OUT_DIR / 'rel_location_practice.csv', 'BELONGS_TO_PRACTICE')


In [ ]:
validate('BELONGS_TO_PRACTICE', len(df), OUT_DIR / 'rel_location_practice.csv')

### 4.3 ISSUED_BY_PRACTICE (InsurancePlan → Practice)


In [ ]:
df = pd.read_parquet(DATA_DIR / '02_dims/insurance.parquet', columns=["Source_Database_Code","PlanNumber"])
out = pd.DataFrame()
out[":START_ID(InsurancePlan)"] = iid(df["Source_Database_Code"], df["PlanNumber"])
out[":END_ID(Practice)"]        = df["Source_Database_Code"]
out[":TYPE"]                    = "ISSUED_BY_PRACTICE"
write_csv(out, OUT_DIR / 'rel_insurance_practice.csv', 'ISSUED_BY_PRACTICE')


In [ ]:
validate('ISSUED_BY_PRACTICE', len(df), OUT_DIR / 'rel_insurance_practice.csv')

### 4.4 RUN_BY (Campaign → Practice)


In [ ]:
df = pd.read_parquet(DATA_DIR / '03_supplementary/campaign_map.parquet').dropna(subset=["Source Database Code"])
out = pd.DataFrame()
out[":START_ID(Campaign)"] = df["Campaign Name"]
out[":END_ID(Practice)"]   = df["Source Database Code"]
out[":TYPE"]               = "RUN_BY"
write_csv(out, OUT_DIR / 'rel_campaign_practice.csv', 'RUN_BY')


In [ ]:
validate('RUN_BY', len(df), OUT_DIR / 'rel_campaign_practice.csv')

### 4.5 REVIEWS (BirdeyeReview → Location)


In [ ]:
be = pd.read_parquet(DATA_DIR / '02_dims/birdeye.parquet').drop_duplicates(subset=["Location","Date Posted On"])
loc = pd.read_parquet(DATA_DIR / '02_dims/location.parquet', columns=["Source_Database_Code","LocationID","LocationName"])
loc["loc_id"] = lid(loc["Source_Database_Code"], loc["LocationID"])
merged = be.merge(loc, left_on="Location", right_on="LocationName", how="inner")
out = pd.DataFrame()
out[":START_ID(BirdeyeReview)"] = merged["Location"].astype(str) + "::" + merged["Date Posted On"].astype(str)
out[":END_ID(Location)"]        = merged["loc_id"]
out[":TYPE"]                    = "REVIEWS"
write_csv(out, OUT_DIR / 'rel_birdeye_location.csv', 'REVIEWS')
print(f"   matched {len(merged):,} / {len(be):,} birdeye reviews to a location")


In [ ]:
validate('REVIEWS', len(merged), OUT_DIR / 'rel_birdeye_location.csv')

### 4.6 HAD_VISIT (Patient → Visit)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/visits.parquet', columns=["Source_Database_Code","PatientID","VisitID"]).drop_duplicates()
out = pd.DataFrame()
out[":START_ID(Patient)"] = pid(df["Source_Database_Code"], df["PatientID"])
out[":END_ID(Visit)"]     = vid(df["Source_Database_Code"], df["VisitID"])
out[":TYPE"]              = "HAD_VISIT"
write_csv(out, OUT_DIR / 'rel_patient_visit.csv', 'HAD_VISIT')


In [ ]:
validate('HAD_VISIT', len(df), OUT_DIR / 'rel_patient_visit.csv')

### 4.7 PERFORMED_AT (Visit → Location)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/visits.parquet', columns=["Source_Database_Code","VisitID","LocationID"]).drop_duplicates(subset=["Source_Database_Code","VisitID"])
out = pd.DataFrame()
out[":START_ID(Visit)"]   = vid(df["Source_Database_Code"], df["VisitID"])
out[":END_ID(Location)"]  = lid(df["Source_Database_Code"], df["LocationID"])
out[":TYPE"]              = "PERFORMED_AT"
write_csv(out, OUT_DIR / 'rel_visit_location.csv', 'PERFORMED_AT')


In [ ]:
validate('PERFORMED_AT', len(df), OUT_DIR / 'rel_visit_location.csv')

### 4.8 UNDER_PLAN (Visit → InsurancePlan)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/visits.parquet',
                     columns=["Source_Database_Code","VisitID","PrimaryInsurancePlanNum"]).drop_duplicates(subset=["Source_Database_Code","VisitID"])
df = df.dropna(subset=["PrimaryInsurancePlanNum"])
df = df[df["PrimaryInsurancePlanNum"].astype(str).str.strip() != ""]
out = pd.DataFrame()
out[":START_ID(Visit)"]       = vid(df["Source_Database_Code"], df["VisitID"])
out[":END_ID(InsurancePlan)"] = iid(df["Source_Database_Code"], df["PrimaryInsurancePlanNum"])
out["plan_type"]              = "primary"
out[":TYPE"]                  = "UNDER_PLAN"
write_csv(out, OUT_DIR / 'rel_visit_insurance.csv', 'UNDER_PLAN')


In [ ]:
validate('UNDER_PLAN', len(df), OUT_DIR / 'rel_visit_insurance.csv')

### 4.9 HAS_CHARGE (Patient → Charge)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet',
                     columns=["Source_Database_Code","PatientID","ChargeID","ChargeAmount","ServiceDate","ProcedureModality"])
out = pd.DataFrame()
out[":START_ID(Patient)"]   = pid(df["Source_Database_Code"], df["PatientID"])
out[":END_ID(Charge)"]      = cid(df["Source_Database_Code"], df["ChargeID"])
out["charge_amount:float"]  = df["ChargeAmount"].fillna("")
out["service_date:datetime"]= fmt_dt(df["ServiceDate"])
out["modality"]             = df["ProcedureModality"].fillna("")
out[":TYPE"]                = "HAS_CHARGE"
write_csv(out, OUT_DIR / 'rel_patient_charge.csv', 'HAS_CHARGE')


In [ ]:
validate('HAS_CHARGE', len(df), OUT_DIR / 'rel_patient_charge.csv')

### 4.10 PART_OF_VISIT (Charge → Visit)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet', columns=["Source_Database_Code","ChargeID","VisitID"])
df = df.dropna(subset=["VisitID"])
df = df[df["VisitID"].astype(str).str.strip() != ""]
out = pd.DataFrame()
out[":START_ID(Charge)"]  = cid(df["Source_Database_Code"], df["ChargeID"])
out[":END_ID(Visit)"]     = vid(df["Source_Database_Code"], df["VisitID"])
out[":TYPE"]              = "PART_OF_VISIT"
write_csv(out, OUT_DIR / 'rel_charge_visit.csv', 'PART_OF_VISIT')


In [ ]:
validate('PART_OF_VISIT', len(df), OUT_DIR / 'rel_charge_visit.csv')

### 4.11 AT_LOCATION (Charge → Location)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet', columns=["Source_Database_Code","ChargeID","LocationID"])
out = pd.DataFrame()
out[":START_ID(Charge)"]  = cid(df["Source_Database_Code"], df["ChargeID"])
out[":END_ID(Location)"]  = lid(df["Source_Database_Code"], df["LocationID"])
out[":TYPE"]              = "AT_LOCATION"
write_csv(out, OUT_DIR / 'rel_charge_location.csv', 'AT_LOCATION')


In [ ]:
validate('AT_LOCATION', len(df), OUT_DIR / 'rel_charge_location.csv')

### 4.12 DIAGNOSED_WITH (Charge → DiagnosisCode)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet',
                     columns=["Source_Database_Code","ChargeID",
                              "ICD10Diagnosis1","ICD10Diagnosis2","ICD10Diagnosis3","ICD10Diagnosis4","ICD10Diagnosis5"])
rows = []
for col in ["ICD10Diagnosis1","ICD10Diagnosis2","ICD10Diagnosis3","ICD10Diagnosis4","ICD10Diagnosis5"]:
    sub = df[["Source_Database_Code","ChargeID",col]].dropna(subset=[col])
    sub = sub[sub[col].astype(str).str.strip() != ""]
    rows.append(pd.DataFrame({
        "start": cid(sub["Source_Database_Code"], sub["ChargeID"]),
        "end":   sub[col].astype(str),
    }))
merged = pd.concat(rows, ignore_index=True).drop_duplicates()
out = pd.DataFrame()
out[":START_ID(Charge)"]      = merged["start"]
out[":END_ID(DiagnosisCode)"] = merged["end"]
out[":TYPE"]                  = "DIAGNOSED_WITH"
write_csv(out, OUT_DIR / 'rel_charge_diagnosis.csv', 'DIAGNOSED_WITH')


In [ ]:
validate('DIAGNOSED_WITH', len(merged), OUT_DIR / 'rel_charge_diagnosis.csv')

### 4.13 USES_PROCEDURE (Charge → ProcedureCode)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet', columns=["Source_Database_Code","ChargeID","ProcedureCode"])
df = df.dropna(subset=["ProcedureCode"])
df = df[df["ProcedureCode"].astype(str).str.strip() != ""]
out = pd.DataFrame()
out[":START_ID(Charge)"]       = cid(df["Source_Database_Code"], df["ChargeID"])
out[":END_ID(ProcedureCode)"]  = df["ProcedureCode"].astype(str)
out[":TYPE"]                   = "USES_PROCEDURE"
write_csv(out, OUT_DIR / 'rel_charge_procedure.csv', 'USES_PROCEDURE')


In [ ]:
validate('USES_PROCEDURE', len(df), OUT_DIR / 'rel_charge_procedure.csv')

### 4.14 SETTLES (Transaction → Charge)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/transactions.parquet',
                     columns=["Source_Database_Code","PaymentID","ChargeID",
                              "PaymentAmount","AdjustmentAmount","AdjustmentBucket",
                              "BadDebtAdjustments","BalanceAfterPost"])
out = pd.DataFrame()
out[":START_ID(Transaction)"]   = tid(df["Source_Database_Code"], df["PaymentID"])
out[":END_ID(Charge)"]          = cid(df["Source_Database_Code"], df["ChargeID"])
out["payment_amount:float"]     = df["PaymentAmount"].fillna("")
out["adjustment_amount:float"]  = df["AdjustmentAmount"].fillna("")
out["adjustment_bucket"]        = df["AdjustmentBucket"].fillna("")
out["bad_debt:float"]           = df["BadDebtAdjustments"].fillna("")
out["balance_after_post:float"] = df["BalanceAfterPost"].fillna("")
out[":TYPE"]                    = "SETTLES"
write_csv(out, OUT_DIR / 'rel_transaction_charge.csv', 'SETTLES')


In [ ]:
validate('SETTLES', len(df), OUT_DIR / 'rel_transaction_charge.csv')

### 4.15 HAS_TRANSACTION (Patient → Transaction)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/transactions.parquet',
                     columns=["Source_Database_Code","PatientID","PaymentID","PaymentAmount","AdjustmentAmount","AdjustmentBucket"])
out = pd.DataFrame()
out[":START_ID(Patient)"]       = pid(df["Source_Database_Code"], df["PatientID"])
out[":END_ID(Transaction)"]     = tid(df["Source_Database_Code"], df["PaymentID"])
out["payment_amount:float"]     = df["PaymentAmount"].fillna("")
out["adjustment_amount:float"]  = df["AdjustmentAmount"].fillna("")
out["bucket"]                   = df["AdjustmentBucket"].fillna("")
out[":TYPE"]                    = "HAS_TRANSACTION"
write_csv(out, OUT_DIR / 'rel_patient_transaction.csv', 'HAS_TRANSACTION')


In [ ]:
validate('HAS_TRANSACTION', len(df), OUT_DIR / 'rel_patient_transaction.csv')

### 4.16 RECEIVED_STATEMENT (Patient → Statement)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/statements.parquet',
                     columns=["Source_Database_Code","PatientID","StatementID","PatientBalance","TotalBalance","StatementLevel"])
out = pd.DataFrame()
out[":START_ID(Patient)"]     = pid(df["Source_Database_Code"], df["PatientID"])
out[":END_ID(Statement)"]     = df["StatementID"]
out["patient_balance:float"]  = df["PatientBalance"].fillna("")
out["total_balance:float"]    = df["TotalBalance"].fillna("")
out["level"]                  = df["StatementLevel"].fillna("")
out[":TYPE"]                  = "RECEIVED_STATEMENT"
write_csv(out, OUT_DIR / 'rel_patient_statement.csv', 'RECEIVED_STATEMENT')


In [ ]:
validate('RECEIVED_STATEMENT', len(df), OUT_DIR / 'rel_patient_statement.csv')

### 4.17 PART_OF_CAMPAIGN (RCCall → Campaign)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/ringcentral.parquet', columns=["Contact_ID","Campaign_Name"]).dropna(subset=["Campaign_Name"])
out = pd.DataFrame()
out[":START_ID(RCCall)"]    = df["Contact_ID"]
out[":END_ID(Campaign)"]    = df["Campaign_Name"]
out[":TYPE"]                = "PART_OF_CAMPAIGN"
write_csv(out, OUT_DIR / 'rel_rccall_campaign.csv', 'PART_OF_CAMPAIGN')


In [ ]:
validate('PART_OF_CAMPAIGN', len(df), OUT_DIR / 'rel_rccall_campaign.csv')

### 4.18 ATTRIBUTED_TO_PHONE (RCCall → PhoneBridge)


In [ ]:
rc = pd.read_parquet(DATA_DIR / '01_facts/ringcentral.parquet',
                       columns=["Contact_ID","ANI_DIALNUM_norm","_rc_attributable"])
rc = rc[rc["_rc_attributable"] == True].dropna(subset=["ANI_DIALNUM_norm"])

pb = pd.read_parquet(DATA_DIR / '03_supplementary/phone_bridge.parquet',
                     columns=["Source_Database_Code","PatientID","phone_norm"]).drop_duplicates()
pb["pb_id"] = pb["Source_Database_Code"].astype(str) + ":" + pb["PatientID"].astype(str) + ":" + pb["phone_norm"].astype(str)

merged = rc.merge(pb, left_on="ANI_DIALNUM_norm", right_on="phone_norm", how="inner")
out = pd.DataFrame()
out[":START_ID(RCCall)"]     = merged["Contact_ID"]
out[":END_ID(PhoneBridge)"]  = merged["pb_id"]
out[":TYPE"]                 = "ATTRIBUTED_TO_PHONE"
write_csv(out, OUT_DIR / 'rel_rccall_phonebridge.csv', 'ATTRIBUTED_TO_PHONE')
print(f"   joined {len(merged):,} rows from rc({len(rc):,}) × pb({len(pb):,})")


In [ ]:
validate('ATTRIBUTED_TO_PHONE', len(merged), OUT_DIR / 'rel_rccall_phonebridge.csv')

### 4.19 IDENTIFIED_BY_PHONE (Patient → PhoneBridge)


In [ ]:
df = pd.read_parquet(DATA_DIR / '03_supplementary/phone_bridge.parquet',
                     columns=["Source_Database_Code","PatientID","phone_norm"]).drop_duplicates()
pbid = df["Source_Database_Code"].astype(str) + ":" + df["PatientID"].astype(str) + ":" + df["phone_norm"].astype(str)
out = pd.DataFrame()
out[":START_ID(Patient)"]   = pid(df["Source_Database_Code"], df["PatientID"])
out[":END_ID(PhoneBridge)"] = pbid
out[":TYPE"]                = "IDENTIFIED_BY_PHONE"
write_csv(out, OUT_DIR / 'rel_patient_phonebridge.csv', 'IDENTIFIED_BY_PHONE')


In [ ]:
validate('IDENTIFIED_BY_PHONE', len(df), OUT_DIR / 'rel_patient_phonebridge.csv')

### 4.20 CALLED_IVR (Patient → IVRInbound)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/rv_inbound.parquet',
                     columns=["ResponseID","AccountID","Balance","AmountPaid","CallDateTime"]).dropna(subset=["AccountID"])
pat = pd.read_parquet(DATA_DIR / '02_dims/patient.parquet', columns=["Source_Database_Code","PatientID"])
pat["PatientID_str"] = pat["PatientID"].astype(str)
df["AccountID_str"]  = df["AccountID"].astype(str).str.replace(r"\.0$","",regex=True)
merged = df.merge(pat, left_on="AccountID_str", right_on="PatientID_str", how="inner")
out = pd.DataFrame()
out[":START_ID(Patient)"]    = pid(merged["Source_Database_Code"], merged["PatientID"])
out[":END_ID(IVRInbound)"]   = merged["ResponseID"]
out["amount_paid:float"]     = merged["AmountPaid"].fillna("")
out["balance:float"]         = merged["Balance"].fillna("")
out["call_date:datetime"]    = fmt_dt(merged["CallDateTime"])
out[":TYPE"]                 = "CALLED_IVR"
write_csv(out, OUT_DIR / 'rel_patient_ivr.csv', 'CALLED_IVR')
print(f"   joined {len(merged):,} IVR rows to a known patient (of {len(df):,} non-null AccountIDs)")


In [ ]:
validate('CALLED_IVR', len(merged), OUT_DIR / 'rel_patient_ivr.csv')

### 4.21 CONTACTED_BY_DIALLER (Patient → DiallerCall)


In [ ]:
df = pd.read_parquet(DATA_DIR / '01_facts/rv_outbound.parquet',
                     columns=["ACCOUNT","ACCOUNTID","PATIENTBALANCE","CALLDATETIME","RESULTSDESC"]).dropna(subset=["ACCOUNTID"])
df["ACCOUNTID_str"] = df["ACCOUNTID"].astype(str).str.replace(r"\.0$","",regex=True)
pat = pd.read_parquet(DATA_DIR / '02_dims/patient.parquet', columns=["Source_Database_Code","PatientID"])
pat["PatientID_str"] = pat["PatientID"].astype(str)
merged = df.merge(pat, left_on="ACCOUNTID_str", right_on="PatientID_str", how="inner")
out = pd.DataFrame()
out[":START_ID(Patient)"]       = pid(merged["Source_Database_Code"], merged["PatientID"])
out[":END_ID(DiallerCall)"]     = merged["ACCOUNT"]
out["patient_balance:float"]    = merged["PATIENTBALANCE"].fillna("")
out["call_date:datetime"]       = fmt_dt(merged["CALLDATETIME"])
out["result"]                   = merged["RESULTSDESC"].fillna("")
out[":TYPE"]                    = "CONTACTED_BY_DIALLER"
write_csv(out, OUT_DIR / 'rel_patient_dialler.csv', 'CONTACTED_BY_DIALLER')


In [ ]:
validate('CONTACTED_BY_DIALLER', len(merged), OUT_DIR / 'rel_patient_dialler.csv')

## 5. Generate `run_import.sh`

After all CSVs are produced and validated, generate the `neo4j-admin` shell
script that bulk-loads them into Neo4j.


In [ ]:
script = """#!/bin/bash
# Auto-generated by parquet_to_neo4j_csv.ipynb
IMPORT_DIR="/var/lib/neo4j/import"
CONTAINER="rp-neo4j"

docker exec $CONTAINER neo4j-admin database import full neo4j \\
  --nodes=Patient=$IMPORT_DIR/nodes_patient.csv \\
  --nodes=Practice=$IMPORT_DIR/nodes_practice.csv \\
  --nodes=Location=$IMPORT_DIR/nodes_location.csv \\
  --nodes=InsurancePlan=$IMPORT_DIR/nodes_insurance.csv \\
  --nodes=Campaign=$IMPORT_DIR/nodes_campaign.csv \\
  --nodes=BirdeyeReview=$IMPORT_DIR/nodes_birdeye.csv \\
  --nodes=Visit=$IMPORT_DIR/nodes_visit.csv \\
  --nodes=Charge=$IMPORT_DIR/nodes_charge.csv \\
  --nodes=Transaction=$IMPORT_DIR/nodes_transaction.csv \\
  --nodes=Statement=$IMPORT_DIR/nodes_statement.csv \\
  --nodes=RCCall=$IMPORT_DIR/nodes_rccall.csv \\
  --nodes=IVRInbound=$IMPORT_DIR/nodes_ivrinbound.csv \\
  --nodes=DiallerCall=$IMPORT_DIR/nodes_diallercall.csv \\
  --nodes=PhoneBridge=$IMPORT_DIR/nodes_phonebridge.csv \\
  --nodes=DiagnosisCode=$IMPORT_DIR/nodes_diagnosiscode.csv \\
  --nodes=ProcedureCode=$IMPORT_DIR/nodes_procedurecode.csv \\
  --relationships=REGISTERED_AT=$IMPORT_DIR/rel_patient_practice.csv \\
  --relationships=BELONGS_TO_PRACTICE=$IMPORT_DIR/rel_location_practice.csv \\
  --relationships=ISSUED_BY_PRACTICE=$IMPORT_DIR/rel_insurance_practice.csv \\
  --relationships=RUN_BY=$IMPORT_DIR/rel_campaign_practice.csv \\
  --relationships=REVIEWS=$IMPORT_DIR/rel_birdeye_location.csv \\
  --relationships=HAD_VISIT=$IMPORT_DIR/rel_patient_visit.csv \\
  --relationships=PERFORMED_AT=$IMPORT_DIR/rel_visit_location.csv \\
  --relationships=UNDER_PLAN=$IMPORT_DIR/rel_visit_insurance.csv \\
  --relationships=HAS_CHARGE=$IMPORT_DIR/rel_patient_charge.csv \\
  --relationships=PART_OF_VISIT=$IMPORT_DIR/rel_charge_visit.csv \\
  --relationships=AT_LOCATION=$IMPORT_DIR/rel_charge_location.csv \\
  --relationships=DIAGNOSED_WITH=$IMPORT_DIR/rel_charge_diagnosis.csv \\
  --relationships=USES_PROCEDURE=$IMPORT_DIR/rel_charge_procedure.csv \\
  --relationships=SETTLES=$IMPORT_DIR/rel_transaction_charge.csv \\
  --relationships=HAS_TRANSACTION=$IMPORT_DIR/rel_patient_transaction.csv \\
  --relationships=RECEIVED_STATEMENT=$IMPORT_DIR/rel_patient_statement.csv \\
  --relationships=PART_OF_CAMPAIGN=$IMPORT_DIR/rel_rccall_campaign.csv \\
  --relationships=ATTRIBUTED_TO_PHONE=$IMPORT_DIR/rel_rccall_phonebridge.csv \\
  --relationships=IDENTIFIED_BY_PHONE=$IMPORT_DIR/rel_patient_phonebridge.csv \\
  --relationships=CALLED_IVR=$IMPORT_DIR/rel_patient_ivr.csv \\
  --relationships=CONTACTED_BY_DIALLER=$IMPORT_DIR/rel_patient_dialler.csv \\
  --skip-bad-relationships=true \\
  --skip-duplicate-nodes=true \\
  --high-io=true \\
  --verbose

docker start $CONTAINER
"""
sh = OUT_DIR / "run_import.sh"
sh.write_text(script)
sh.chmod(0o755)
print(f"Wrote {sh}")
print(f"\nTotal CSVs in {OUT_DIR}:")
total_mb = 0
for p in sorted(OUT_DIR.glob('*.csv')):
    mb = p.stat().st_size / 1024 / 1024
    total_mb += mb
    print(f"  {p.name:<40} {mb:>8.2f} MB")
print(f"  {'TOTAL':<40} {total_mb:>8.2f} MB")


## 6. Post-import: compare Neo4j vs source parquets

Connect to the live Neo4j container and verify the loaded graph matches the
parquets. Requires `neo4j` Python driver (`pip install neo4j`).


In [ ]:
from neo4j import GraphDatabase

NEO4J_URI      = 'bolt://localhost:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'rp_strong_pass_2025'  # match dockers/.env

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as s:
    rec = s.run("RETURN 'connected' AS msg").single()
print(rec['msg'])


In [ ]:
def graph_count(cypher):
    with driver.session() as s:
        return s.run(cypher).single().value()

EXPECTED = {
    'Patient':       len(pd.read_parquet(DATA_DIR / '02_dims/patient.parquet', columns=['PatientID'])),
    'Practice':      pd.read_parquet(DATA_DIR / '02_dims/patient.parquet', columns=['Source_Database_Code'])['Source_Database_Code'].nunique(),
    'Location':      len(pd.read_parquet(DATA_DIR / '02_dims/location.parquet', columns=['LocationID'])),
    'InsurancePlan': len(pd.read_parquet(DATA_DIR / '02_dims/insurance.parquet', columns=['PlanNumber'])),
    'Visit':         pd.read_parquet(DATA_DIR / '01_facts/visits.parquet', columns=['Source_Database_Code','VisitID']).drop_duplicates().shape[0],
    'Charge':        len(pd.read_parquet(DATA_DIR / '01_facts/charges.parquet', columns=['ChargeID'])),
    'Transaction':   len(pd.read_parquet(DATA_DIR / '01_facts/transactions.parquet', columns=['PaymentID'])),
    'Statement':     len(pd.read_parquet(DATA_DIR / '01_facts/statements.parquet', columns=['StatementID'])),
    'RCCall':        len(pd.read_parquet(DATA_DIR / '01_facts/ringcentral.parquet', columns=['Contact_ID'])),
    'IVRInbound':    len(pd.read_parquet(DATA_DIR / '01_facts/rv_inbound.parquet', columns=['ResponseID'])),
    'DiallerCall':   len(pd.read_parquet(DATA_DIR / '01_facts/rv_outbound.parquet', columns=['ACCOUNT'])),
}

print(f"{'Label':<20} {'Expected':>12} {'Neo4j':>12} {'Δ':>10}")
print("-" * 56)
for label, exp in EXPECTED.items():
    actual = graph_count(f"MATCH (n:{label}) RETURN count(n)")
    delta = actual - exp
    flag = "✓" if delta == 0 else ("≈" if abs(delta) < 5 else "✗")
    print(f"{flag} {label:<18} {exp:>12,} {actual:>12,} {delta:>+10,}")


In [ ]:
# Relationship counts vs source row counts
REL_EXPECTED = {
    'REGISTERED_AT':       len(pd.read_parquet(DATA_DIR / '02_dims/patient.parquet', columns=['PatientID'])),
    'BELONGS_TO_PRACTICE': len(pd.read_parquet(DATA_DIR / '02_dims/location.parquet', columns=['LocationID'])),
    'ISSUED_BY_PRACTICE':  len(pd.read_parquet(DATA_DIR / '02_dims/insurance.parquet', columns=['PlanNumber'])),
    'HAD_VISIT':           pd.read_parquet(DATA_DIR / '01_facts/visits.parquet', columns=['Source_Database_Code','PatientID','VisitID']).drop_duplicates().shape[0],
    'PART_OF_VISIT':       pd.read_parquet(DATA_DIR / '01_facts/charges.parquet', columns=['VisitID']).dropna().pipe(lambda d: d[d['VisitID'].astype(str).str.strip() != '']).shape[0],
    'AT_LOCATION':         len(pd.read_parquet(DATA_DIR / '01_facts/charges.parquet', columns=['ChargeID'])),
    'SETTLES':             len(pd.read_parquet(DATA_DIR / '01_facts/transactions.parquet', columns=['PaymentID'])),
    'HAS_TRANSACTION':     len(pd.read_parquet(DATA_DIR / '01_facts/transactions.parquet', columns=['PaymentID'])),
    'RECEIVED_STATEMENT':  len(pd.read_parquet(DATA_DIR / '01_facts/statements.parquet', columns=['StatementID'])),
}

print(f"{'Type':<24} {'Expected':>12} {'Neo4j':>12} {'Δ':>10}")
print("-" * 60)
for rtype, exp in REL_EXPECTED.items():
    actual = graph_count(f"MATCH ()-[r:{rtype}]->() RETURN count(r)")
    delta = actual - exp
    flag = "✓" if delta == 0 else ("≈" if abs(delta) < 5 else "✗")
    print(f"{flag} {rtype:<22} {exp:>12,} {actual:>12,} {delta:>+10,}")


In [ ]:
# Spot-check: pick one patient, verify their charges + visits match parquet
SAMPLE_PATIENT = None  # set to e.g. "SAPA:1000000" to pin a specific patient

with driver.session() as s:
    if SAMPLE_PATIENT is None:
        SAMPLE_PATIENT = s.run("MATCH (p:Patient) RETURN p.`patientId:ID(Patient)` AS id LIMIT 1").single()
        if SAMPLE_PATIENT is None:
            # Fallback: column name in graph is plain "patientId"
            SAMPLE_PATIENT = s.run("MATCH (p:Patient) RETURN p.patientId AS id LIMIT 1").single()
        SAMPLE_PATIENT = SAMPLE_PATIENT['id']
print(f"Sample patient: {SAMPLE_PATIENT}")

src_db, pid_str = SAMPLE_PATIENT.split(":", 1)
parquet_charges = pd.read_parquet(DATA_DIR / '01_facts/charges.parquet',
    columns=['Source_Database_Code','PatientID','ChargeID'])
parquet_charges = parquet_charges[(parquet_charges['Source_Database_Code'] == src_db) &
                                  (parquet_charges['PatientID'].astype(str) == pid_str)]
print(f"Parquet: {len(parquet_charges)} charges for patient {SAMPLE_PATIENT}")

with driver.session() as s:
    graph_charges = s.run("""
        MATCH (p:Patient {patientId:$pid})-[:HAS_CHARGE]->(c:Charge)
        RETURN count(c) AS n
    """, pid=SAMPLE_PATIENT).single()['n']
print(f"Neo4j  : {graph_charges} HAS_CHARGE relationships")
print("✓ Match" if len(parquet_charges) == graph_charges else "✗ Mismatch")


In [ ]:
driver.close()